# Easy Agent — The `Skill` Tool Subsystem

A self-contained Python walkthrough of how `SKILL.md` files on disk become
callable, prompt-driven workflows the model can invoke at runtime.

Every class and function below is inlined from the TypeScript source
(`src/services/skills/`, `src/tools/skillTool.ts`, `src/tools/Tool.ts`,
`src/types/types.ts`, `src/core/queryEngine.ts`). No project imports are used.

| Notebook section | TypeScript source |
|---|---|
| 2. Data types | `src/types/types.ts` |
| 3. Tool interface (minimal) | `src/tools/Tool.ts` |
| 4. Frontmatter parsing | `src/services/skills/parseFrontmatter.ts` |
| 5. Disk loader | `src/services/skills/loadSkillsDir.ts` |
| 6. Registry (two-map) | `src/services/skills/registry.ts` |
| 7. Budget & listing | `src/services/skills/budget.ts` |
| 8. Bootstrap | `src/services/skills/bootstrap.ts` |
| 9. Conditional activation | `src/services/skills/conditional.ts` |
| 10. The `Skill` tool | `src/tools/skillTool.ts` |
| 11. User slash-command expansion | `src/core/queryEngine.ts` |
| 12. End-to-end demo | (assembled) |

**Excluded for clarity:** UI rendering, MCP integration, permission settings
loading, sandbox wiring, observability hooks, full agentic-loop plumbing,
and the `disable-model-invocation` / `context: fork` guards beyond the
minimum shown.


## 1. Imports, Path Discovery & Temp Workspace

In [ ]:
import os, re, json, asyncio, shutil, tempfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Optional, Literal

import yaml  # PyYAML — for frontmatter parsing

# ─── Path discovery — walk up to find the project root ────
_cwd = Path('.').resolve()
PROJECT_ROOT = _cwd
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / 'pyproject.toml').exists() or (PROJECT_ROOT / 'package.json').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

# Skill discovery scopes (mirrors src/utils/paths.ts).
def get_user_skills_dir() -> Path:
    """~/.easy-agent/skills/ — user-scope skills."""
    return Path.home() / '.easy-agent' / 'skills'

def get_project_skills_dir(cwd: Path) -> Path:
    """<cwd>/.easy-agent/skills/ — project-scope skills (override user)."""
    return Path(cwd) / '.easy-agent' / 'skills'

# We'll mostly demo against an isolated temp workspace so nothing on disk
# leaks into the notebook. We override the user-dir getter in the demo cells.
DEMO_WORKSPACE = Path(tempfile.mkdtemp(prefix='easy-agent-skill-demo-'))
print(f'DEMO_WORKSPACE: {DEMO_WORKSPACE}')


## 2. Data Types — `Skill`, `SkillFrontmatter`, `SkillSource`

The Skill subsystem revolves around three shapes:

- **`SkillSource`** — where the skill came from (`user` or `project`); used
  for override priority and display.
- **`SkillFrontmatter`** — the parsed + normalized YAML metadata block.
  Carries the fields that influence runtime behavior (`paths`,
  `allowed-tools`, `disable-model-invocation`, …).
- **`Skill`** — the fully-loaded, ready-to-invoke object combining the
  Markdown body with its metadata and resolved filesystem paths.


In [ ]:
SkillSource = Literal['user', 'project']

@dataclass
class SkillFrontmatter:
    """Mirrors src/types/types.ts:SkillFrontmatter."""
    name: Optional[str] = None
    description: Optional[str] = None
    when_to_use: Optional[str] = None
    allowedTools: list[str] = field(default_factory=list)
    argumentHint: Optional[str] = None
    disableModelInvocation: bool = False
    paths: Optional[list[str]] = None        # gitignore-style — triggers conditional activation
    hasForkContext: bool = False             # `context: fork` — currently rejected at exec
    raw: dict[str, Any] = field(default_factory=dict)  # untouched YAML for forward compat

@dataclass
class Skill:
    """Mirrors src/types/types.ts:Skill."""
    name: str
    description: str
    body: str
    filePath: str
    baseDir: str
    source: SkillSource
    frontmatter: SkillFrontmatter
    whenToUse: Optional[str] = None


## 3. Tool Interface — `Tool`, `ToolContext`, `ToolResult`

Every agent capability — including `Skill` itself — implements the same `Tool`
interface. We inline the minimum needed to make the `Skill` tool runnable
(name, description, JSON schema, `call()`, `is_read_only`, `is_enabled`).

**Simplified vs. source (`src/tools/Tool.ts`):** dropped MCP/permissions
wiring, concurrency-safety flag, result-size truncation, and sub-agent
spawning context fields. The Skill tool only uses `cwd`, `sessionId`, and
the `addSessionAllowRules` callback.


In [ ]:
@dataclass
class ToolContext:
    """Runtime context passed to every tool invocation."""
    cwd: str
    sessionId: Optional[str] = None
    # Callback that lets a tool whitelist other tools for this session
    # (so subsequent calls don't re-prompt for permission). The Skill
    # tool calls this with the skill's `allowed-tools` frontmatter list.
    addSessionAllowRules: Optional[Any] = None  # Callable[[list[str]], None]

@dataclass
class ToolResult:
    content: str
    isError: bool = False

class Tool:
    """Minimal Tool ABC — name + inputSchema + call() + read-only/enabled."""
    name: str = ''
    description: str = ''
    inputSchema: dict[str, Any] = {}
    async def call(self, input: dict[str, Any], context: ToolContext) -> ToolResult: ...
    def is_read_only(self) -> bool: return True
    def is_enabled(self) -> bool: return True


## 4. Frontmatter Parsing — `splitFrontmatter` + `normalizeFrontmatter`

Source: `src/services/skills/parseFrontmatter.ts`.

Two-phase parse:

1. **`split_frontmatter()`** — regex-split the `---\n<yaml>\n---\n<body>`
   document. Returns `{raw, body, parse_error?}`. Invalid YAML is reported
   via `parse_error` rather than raising (so one bad SKILL.md doesn't crash
   startup).
2. **`normalize_frontmatter()`** — coerce raw YAML keys into the typed
   `SkillFrontmatter` dataclass. Supports both kebab-case (`allowed-tools`)
   and camelCase (`allowedTools`).

A fallback extractor (`extract_fallback_description`) pulls the first
non-empty paragraph from the body when no `description:` field exists.


In [ ]:
@dataclass
class FrontmatterSplit:
    raw: dict[str, Any]
    body: str
    parse_error: Optional[str] = None

FRONTMATTER_RE = re.compile(r'^---\r?\n([\s\S]*?)\r?\n---\r?\n?([\s\S]*)$')

def split_frontmatter(content: str) -> FrontmatterSplit:
    match = FRONTMATTER_RE.match(content)
    if not match:
        return FrontmatterSplit(raw={}, body=content)
    yaml_text, body = match.group(1), match.group(2)
    try:
        parsed = yaml.safe_load(yaml_text)
    except yaml.YAMLError as e:
        return FrontmatterSplit(raw={}, body=body, parse_error=str(e))
    if isinstance(parsed, dict):
        return FrontmatterSplit(raw=parsed, body=body)
    return FrontmatterSplit(
        raw={}, body=body,
        parse_error='Frontmatter must be a YAML mapping (key: value)',
    )

# ─── Type-coercion helpers (replicate asString / asStringArray / asBoolean)
def _as_string(value: Any) -> Optional[str]:
    if isinstance(value, str):
        v = value.strip()
        return v if v else None
    if isinstance(value, (int, float, bool)):
        return str(value)
    return None

def _as_string_array(value: Any) -> list[str]:
    if isinstance(value, list):
        return [v.strip() for v in value if isinstance(v, str) and v.strip()]
    if isinstance(value, str):
        return [s.strip() for s in value.split(',') if s.strip()]
    return []

def _as_boolean(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() in ('true', 'yes', '1')
    return False


In [ ]:
def extract_fallback_description(body: str) -> str:
    """First non-empty paragraph, skipping leading H1/H2 headings."""
    buf: list[str] = []
    for raw_line in body.splitlines():
        line = raw_line.strip()
        if not line:
            if buf:
                break
            continue
        if not buf and line.startswith('#'):
            continue
        buf.append(line)
    return re.sub(r'\s+', ' ', ' '.join(buf)).strip()

def normalize_frontmatter(raw: dict[str, Any], body: str) -> SkillFrontmatter:
    allowed = _as_string_array(raw.get('allowed-tools') or raw.get('allowedTools'))
    paths = _as_string_array(raw.get('paths'))
    return SkillFrontmatter(
        name=_as_string(raw.get('name')),
        description=_as_string(raw.get('description')),
        when_to_use=_as_string(raw.get('when_to_use') or raw.get('whenToUse')),
        allowedTools=allowed,
        argumentHint=_as_string(raw.get('argument-hint') or raw.get('argumentHint')),
        disableModelInvocation=_as_boolean(
            raw.get('disable-model-invocation') or raw.get('disableModelInvocation'),
        ),
        paths=paths if paths else None,
        hasForkContext=(_as_string(raw.get('context')) == 'fork'),
        raw=raw,
    )


### 4.1 Demo — parse a SKILL.md string

Showcases the split + normalize flow on a representative skill file.


In [ ]:
SAMPLE_SKILL = '''---
name: code-reviewer
description: Reviews recent diffs for bugs and style issues
when_to_use: After a feature implementation, before a PR
allowed-tools:
  - Read
  - Grep
  - Glob
argument-hint: <file-or-dir>
---

# Code Reviewer

Review the files under $ARGUMENTS. Look for:
1. Off-by-one errors
2. Unhandled error cases
3. Naming inconsistencies

Base directory: ${CLAUDE_SKILL_DIR}
Session: ${CLAUDE_SESSION_ID}
'''

split = split_frontmatter(SAMPLE_SKILL)
fm = normalize_frontmatter(split.raw, split.body)
print('parse_error =', split.parse_error)
print('name        =', fm.name)
print('description =', fm.description)
print('when_to_use =', fm.when_to_use)
print('allowedTools =', fm.allowedTools)
print('paths       =', fm.paths)
print('disableModelInvocation =', fm.disableModelInvocation)
print('body (first 80 chars) =', split.body[:80])


## 5. Disk Loader — `loadFromOneDir` + `loadAllSkills`

Source: `src/services/skills/loadSkillsDir.ts`.

Two scopes are scanned **in parallel** and merged with project > user precedence:

| Scope | Path | Precedence |
|---|---|---|
| User | `~/.easy-agent/skills/` | Lower |
| Project | `<cwd>/.easy-agent/skills/` | Higher |

Each subdirectory is one skill; its `SKILL.md` is parsed and merged.
Symlinked duplicates are dropped via `realpath()` (Python: `Path.resolve()`).
Name collisions: project wins because it is loaded second into a Map keyed
by skill name.


In [ ]:
SKILL_FILE = 'SKILL.md'

@dataclass
class LoadedFromDir:
    skills: list[Skill]
    warnings: list[str]

async def load_from_one_dir(dir_path: Path, source: SkillSource) -> LoadedFromDir:
    if not dir_path.exists():
        return LoadedFromDir(skills=[], warnings=[])
    if not dir_path.is_dir():
        return LoadedFromDir(skills=[], warnings=[f'Failed to read {dir_path}: not a directory'])

    out: list[Skill] = []
    warnings: list[str] = []
    for entry in sorted(dir_path.iterdir()):
        if not entry.is_dir():
            continue
        skill_dir = entry
        file_path = skill_dir / SKILL_FILE
        if not file_path.exists():
            continue
        try:
            raw = file_path.read_text(encoding='utf-8')
        except OSError as e:
            warnings.append(f'[skills] Skipping {skill_dir}: {e}')
            continue

        split = split_frontmatter(raw)
        if split.parse_error:
            warnings.append(f'[skills] Skipping {entry.name}: invalid frontmatter ({split.parse_error})')
            continue
        fm = normalize_frontmatter(split.raw, split.body)

        # Resolve symlinks for dedup
        real_file = str(file_path.resolve())
        real_dir = str(skill_dir.resolve())

        name = fm.name or entry.name
        description = fm.description or extract_fallback_description(split.body) or name

        out.append(Skill(
            name=name,
            description=description,
            whenToUse=fm.when_to_use,
            body=split.body,
            filePath=real_file,
            baseDir=real_dir,
            source=source,
            frontmatter=fm,
        ))
    return LoadedFromDir(skills=out, warnings=warnings)

@dataclass
class LoadAllSkillsResult:
    skills: list[Skill]
    warnings: list[str]

async def load_all_skills(cwd: Path, user_dir: Optional[Path] = None) -> LoadAllSkillsResult:
    user = user_dir if user_dir is not None else get_user_skills_dir()
    project = get_project_skills_dir(cwd)
    user_res, project_res = await asyncio.gather(
        load_from_one_dir(user, 'user'),
        load_from_one_dir(project, 'project'),
    )

    seen_real_paths: set[str] = set()
    by_name: dict[str, Skill] = {}
    # User first, then project — project overwrites on name collision.
    for skill in [*user_res.skills, *project_res.skills]:
        if skill.filePath in seen_real_paths:
            continue  # symlink dedup
        seen_real_paths.add(skill.filePath)
        by_name[skill.name] = skill
    return LoadAllSkillsResult(
        skills=list(by_name.values()),
        warnings=[*user_res.warnings, *project_res.warnings],
    )


### 5.1 Demo — write three SKILL.md files and load them

We materialize a tiny user + project skill tree under `DEMO_WORKSPACE`,
then exercise the loader. Note `name-collision` exists in both scopes;
the project version should win.


In [ ]:
DEMO_USER = DEMO_WORKSPACE / 'user-home' / '.easy-agent' / 'skills'
DEMO_PROJECT = DEMO_WORKSPACE / 'project' / '.easy-agent' / 'skills'
DEMO_CWD = DEMO_WORKSPACE / 'project'

def write_skill(parent: Path, dirname: str, content: str) -> None:
    p = parent / dirname
    p.mkdir(parents=True, exist_ok=True)
    (p / 'SKILL.md').write_text(content, encoding='utf-8')

# 1. user-scope skill — visible only
write_skill(DEMO_USER, 'tidy-imports', '''---
name: tidy-imports
description: Sort and dedupe imports in changed files
allowed-tools: [Read, Edit]
---
Run isort over $ARGUMENTS, then verify the file still parses.
''')

# 2. user-scope skill that will be overridden by project
write_skill(DEMO_USER, 'name-collision', '''---
name: name-collision
description: USER VERSION — should be overridden
---
User scope body.
''')

# 3. project-scope override + a project-only conditional skill
write_skill(DEMO_PROJECT, 'name-collision', '''---
name: name-collision
description: PROJECT VERSION — wins
---
Project scope body.
''')

write_skill(DEMO_PROJECT, 'test-reviewer', '''---
name: test-reviewer
description: Reviews test files for coverage gaps
paths:
  - "**/*.test.ts"
  - "**/*.spec.py"
  - "tests/**"
---
Check coverage on the test files just touched.
''')

result = await load_all_skills(DEMO_CWD, user_dir=DEMO_USER)
for s in result.skills:
    print(f'{s.source:8s} {s.name:18s} → {s.description}')
    if s.frontmatter.paths:
        print(f'           ↳ conditional on: {s.frontmatter.paths}')
print('warnings:', result.warnings)


## 6. Registry — Dynamic vs. Conditional Maps

Source: `src/services/skills/registry.ts`.

Two `dict`s partition the loaded skills:

- **`dynamic`** — always visible to the model (no `paths` declared, or
  already promoted). These appear in every system-prompt listing.
- **`conditional`** — declared with `paths`; hidden until a tool touches a
  matching file. Activation is **one-way and sticky** for the process.

Visibility matrix (from `03a_skill_tool.md` §8.2):

| State | `disable-model-invocation` | `paths` | In system prompt? | Model can invoke? | User `/name`? |
|---|---|---|---|---|---|
| Dynamic visible | false | none | ✅ | ✅ | ✅ |
| Dynamic hidden  | true  | none | ❌ | ❌ (rejected) | ✅ |
| Conditional latent | any | yes (unmatched) | ❌ | ❌ (rejected) | ✅ |
| Promoted (conditional → dynamic) | false | yes (matched) | ✅ | ✅ | ✅ |


In [ ]:
class SkillRegistry:
    """Two-map in-memory store. In the real codebase this is module-level state
    in registry.ts; we wrap it in a class so the notebook can spawn multiple
    isolated registries without leaking.
    """
    def __init__(self) -> None:
        self.dynamic: dict[str, Skill] = {}
        self.conditional: dict[str, Skill] = {}
        self.initialized: bool = False

    def set_skills(self, skills: list[Skill]) -> None:
        self.dynamic.clear()
        self.conditional.clear()
        for s in skills:
            if s.frontmatter.paths:
                self.conditional[s.name] = s
            else:
                self.dynamic[s.name] = s
        self.initialized = True

    def find_skill(self, name: str) -> Optional[Skill]:
        return self.dynamic.get(name) or self.conditional.get(name)

    def get_model_visible_skills(self) -> list[Skill]:
        """Used by buildSystemPrompt() — excludes disable-model-invocation."""
        return [s for s in self.dynamic.values() if not s.frontmatter.disableModelInvocation]

    def get_all_user_invocable_skills(self) -> list[Skill]:
        """Used by /command tab-completion — includes hidden + conditional."""
        return [*self.dynamic.values(), *self.conditional.values()]

    def activate_conditional(self, name: str) -> bool:
        """Promote conditional → dynamic. Returns True if it was latent."""
        skill = self.conditional.pop(name, None)
        if skill is None:
            return False
        self.dynamic[name] = skill
        return True

    def list_conditional_skills(self) -> list[Skill]:
        return list(self.conditional.values())

REGISTRY = SkillRegistry()
REGISTRY.set_skills(result.skills)
print('dynamic     :', list(REGISTRY.dynamic.keys()))
print('conditional :', list(REGISTRY.conditional.keys()))
print('visible     :', [s.name for s in REGISTRY.get_model_visible_skills()])
print('find tidy-imports :', REGISTRY.find_skill('tidy-imports').name if REGISTRY.find_skill('tidy-imports') else None)
print('find test-reviewer:', REGISTRY.find_skill('test-reviewer').name if REGISTRY.find_skill('test-reviewer') else None)


## 7. Budget & System-Prompt Listing

Source: `src/services/skills/budget.ts`.

Every system prompt embeds a `<system-reminder>` block listing visible
skills as `- <name>: <description>` lines. The budget system prevents this
from ballooning when a project has dozens of skills.

**Three-tier degradation**, evaluated in order:

1. **Tier 1** — full descriptions (capped at `MAX_LISTING_DESC_CHARS = 250`).
2. **Tier 2** — distribute remaining budget evenly across all skills, but
   ≥ `MIN_DESC_CHARS_PER_SKILL = 20` each.
3. **Tier 3** — names only.

Budget defaults to 8000 chars (~2000 tokens). Override via the
`EASY_AGENT_SKILL_CHAR_BUDGET` env var.


In [ ]:
MAX_LISTING_DESC_CHARS = 250
MIN_DESC_CHARS_PER_SKILL = 20
DEFAULT_BUDGET_CHARS = 8000

def get_skill_char_budget() -> int:
    env = os.environ.get('EASY_AGENT_SKILL_CHAR_BUDGET')
    if env:
        try:
            parsed = int(env)
            if parsed > 0:
                return parsed
        except ValueError:
            pass
    return DEFAULT_BUDGET_CHARS

def _truncate_desc(desc: str, max_chars: int) -> str:
    if len(desc) <= max_chars:
        return desc
    if max_chars <= 1:
        return '…'
    return desc[: max_chars - 1].rstrip() + '…'

def _build_line(skill: Skill, desc_max: int) -> str:
    capped = min(desc_max, MAX_LISTING_DESC_CHARS)
    full = f'{skill.description} — {skill.whenToUse}' if skill.whenToUse else skill.description
    return f'- {skill.name}: {_truncate_desc(full, capped)}'

def format_skills_within_budget(skills: list[Skill], budget: Optional[int] = None) -> str:
    if not skills:
        return ''
    if budget is None:
        budget = get_skill_char_budget()

    # Tier 1: full descriptions
    tier1 = [_build_line(s, MAX_LISTING_DESC_CHARS) for s in skills]
    if sum(len(l) + 1 for l in tier1) <= budget:
        return '\n'.join(tier1)

    # Tier 2: evenly distributed descriptions
    prefix_cost = sum(len(f'- {s.name}: ') + 1 for s in skills)
    desc_budget = budget - prefix_cost
    if desc_budget >= len(skills) * MIN_DESC_CHARS_PER_SKILL:
        per_desc = max(MIN_DESC_CHARS_PER_SKILL, desc_budget // len(skills))
        tier2 = [_build_line(s, per_desc) for s in skills]
        if sum(len(l) + 1 for l in tier2) <= budget:
            return '\n'.join(tier2)

    # Tier 3: names only
    return '\n'.join(f'- {s.name}' for s in skills)

def format_skills_system_reminder(skills: list[Skill]) -> str:
    if not skills:
        return ''
    listing = format_skills_within_budget(skills)
    if not listing:
        return ''
    return '\n'.join([
        '<system-reminder>',
        'Available skills you can invoke via the `Skill` tool. Each line is `- <name>: <description>`.',
        'Call `Skill(skill="<name>", args="<optional args>")` when the user\'s request matches one of these.',
        '',
        listing,
        '</system-reminder>',
    ])


### 7.1 Demo — render the system-reminder listing

Tier 1 case (default budget, only a couple of skills). We then shrink the
budget to force Tier 2, then Tier 3.


In [ ]:
visible = REGISTRY.get_model_visible_skills()
print('───── tier 1 (default 8000-char budget) ─────')
print(format_skills_system_reminder(visible))

print('\n───── tier 2 (forced 100-char budget) ─────')
print(format_skills_within_budget(visible, budget=100))

print('\n───── tier 3 (forced 25-char budget) ─────')
print(format_skills_within_budget(visible, budget=25))


## 8. Bootstrap — `bootstrapSkills(cwd)`

Source: `src/services/skills/bootstrap.ts`.

One-shot startup orchestration: load → populate registry → emit warnings.
Called once from `cli.ts` before the React UI mounts so the first
system prompt already has the discovery listing.


In [ ]:
@dataclass
class SkillsBootstrapResult:
    skillCount: int
    conditionalCount: int
    warnings: list[str]

async def bootstrap_skills(cwd: Path, registry: SkillRegistry, user_dir: Optional[Path] = None) -> SkillsBootstrapResult:
    res = await load_all_skills(cwd, user_dir=user_dir)
    registry.set_skills(res.skills)
    conditional_count = sum(1 for s in res.skills if s.frontmatter.paths)
    for w in res.warnings:
        print(f'[easy-agent] {w}')
    return SkillsBootstrapResult(
        skillCount=len(res.skills) - conditional_count,
        conditionalCount=conditional_count,
        warnings=res.warnings,
    )

# Re-run via bootstrap to show the entry-point shape callers actually see.
REGISTRY = SkillRegistry()
boot_result = await bootstrap_skills(DEMO_CWD, REGISTRY, user_dir=DEMO_USER)
print(f'\nLoaded {boot_result.skillCount} active + {boot_result.conditionalCount} conditional skills')


## 9. Conditional Activation — `activateConditionalSkillsForPaths`

Source: `src/services/skills/conditional.ts`.

After every successful tool call, the agentic loop:

1. Calls `extract_tool_file_paths(tool_name, input)` to pull paths from
   well-known fields (`file_path` for Read/Write/Edit, `path` for Glob).
2. Calls `activate_conditional_skills_for_paths(paths, cwd)`, which
   gitignore-matches each path against every conditional skill's `paths`
   patterns and promotes the matching skills.

**Key properties:**

- **One-way** — once activated, a skill stays in `dynamic` for the process.
- **Sticky** — prevents flicker as the model navigates files.
- Uses gitignore semantics (the TS source uses the `ignore` npm package).
  Here we use `pathspec` (`GitWildMatchPattern`), the Python equivalent.


In [ ]:
# Try pathspec for true gitignore semantics; fall back to a tiny matcher.
try:
    import pathspec  # type: ignore
    _HAS_PATHSPEC = True
except ImportError:
    _HAS_PATHSPEC = False
    print('NOTE: pathspec not installed — `uv pip install pathspec` for accurate gitignore matching')

import fnmatch

def _matches_any(patterns: list[str], rel_paths: list[str]) -> bool:
    if _HAS_PATHSPEC:
        spec = pathspec.PathSpec.from_lines('gitwildmatch', patterns)
        return any(spec.match_file(p) for p in rel_paths)
    # Fallback: translate `**/` to `*` and use fnmatch (lossy but workable).
    fallback = [p.replace('**/', '').replace('/**', '/*') for p in patterns]
    return any(
        any(fnmatch.fnmatch(rel, pat) or fnmatch.fnmatch(rel, '*/' + pat) for pat in fallback)
        for rel in rel_paths
    )

def activate_conditional_skills_for_paths(
    file_paths: list[str], cwd: Path, registry: SkillRegistry,
) -> list[str]:
    if not file_paths:
        return []
    candidates = registry.list_conditional_skills()
    if not candidates:
        return []

    # Convert to repo-relative POSIX paths (gitignore patterns expect that form).
    rel_paths: list[str] = []
    for p in file_paths:
        abs_path = Path(p) if Path(p).is_absolute() else (cwd / p)
        try:
            rel = abs_path.resolve().relative_to(cwd.resolve())
        except ValueError:
            continue  # outside repo — skip
        rel_paths.append(rel.as_posix())
    if not rel_paths:
        return []

    activated: list[str] = []
    for skill in candidates:
        patterns = skill.frontmatter.paths or []
        if not patterns:
            continue
        if _matches_any(patterns, rel_paths):
            if registry.activate_conditional(skill.name):
                activated.append(skill.name)
    return activated

def extract_tool_file_paths(tool_name: str, input: dict[str, Any]) -> list[str]:
    """Conservative extractor — only well-known path fields, to avoid false positives."""
    paths: list[str] = []
    if tool_name in ('Read', 'Write', 'Edit'):
        fp = input.get('file_path')
        if isinstance(fp, str):
            paths.append(fp)
    elif tool_name == 'Glob':
        root = input.get('path')
        if isinstance(root, str):
            paths.append(root)
    return paths


### 9.1 Demo — simulate a tool call that activates `test-reviewer`

Before: `test-reviewer` is in the `conditional` map. We mimic the agentic
loop's post-tool-call hook: extract paths → try activation → check registry.


In [ ]:
print('BEFORE: dynamic =', list(REGISTRY.dynamic.keys()))
print('BEFORE: conditional =', list(REGISTRY.conditional.keys()))

# Pretend the model just called Read(file_path='tests/test_login.spec.py')
tool_name = 'Read'
tool_input = {'file_path': 'tests/test_login.spec.py'}
paths = extract_tool_file_paths(tool_name, tool_input)
print('\nExtracted paths:', paths)

activated = activate_conditional_skills_for_paths(paths, DEMO_CWD, REGISTRY)
print('Activated skills :', activated)

print('\nAFTER : dynamic =', list(REGISTRY.dynamic.keys()))
print('AFTER : conditional =', list(REGISTRY.conditional.keys()))
print('test-reviewer would now appear in the next system-prompt listing.')


## 10. The `Skill` Tool — Lookup, Guard, Substitute, Return

Source: `src/tools/skillTool.ts`.

When the model emits `tool_use { name: 'Skill', input: { skill, args } }`,
the agentic loop dispatches to `skill_tool.call()`. Steps:

1. **Validate name** against `/^[a-zA-Z0-9_-]+$/`.
2. **Lookup** in the registry (dynamic OR conditional).
3. **Guard**: `disable-model-invocation` → reject; `context: fork` → reject.
4. **Inject allowed-tools** into the session permission allow-rules.
5. **Substitute variables** in the body: `$ARGUMENTS`, `${CLAUDE_SKILL_DIR}`,
   `${CLAUDE_SESSION_ID}`.
6. **Return** the substituted body as the tool result — the model reads it
   and continues following the instructions.

Properties:

- `is_read_only() = False` — side-effecting; Plan Mode rejects.
- `is_concurrency_safe()` unset/false — mutates session state.


In [ ]:
SKILL_NAME_RE = re.compile(r'^[a-zA-Z0-9_-]+$')

def _read_input(input: dict[str, Any]) -> tuple[str, str]:
    skill = input.get('skill', '')
    skill = skill.strip() if isinstance(skill, str) else ''
    args = input.get('args', '')
    args = args if isinstance(args, str) else ''
    return skill, args

def substitute_variables(body: str, skill: Skill, args: str, session_id: str) -> str:
    """Three template variables. Order matters slightly — ${CLAUDE_*} first."""
    dir_posix = skill.baseDir.replace(os.sep, '/')
    return (body
            .replace('${CLAUDE_SKILL_DIR}', dir_posix)
            .replace('${CLAUDE_SESSION_ID}', session_id)
            .replace('$ARGUMENTS', args))

def build_prompt_text(skill: Skill, args: str, session_id: str) -> str:
    dir_posix = skill.baseDir.replace(os.sep, '/')
    header = f'Base directory for this skill: {dir_posix}\n\n'
    return header + substitute_variables(skill.body, skill, args, session_id)

class SkillTool(Tool):
    name = 'Skill'
    description = (
        "Execute a named skill within the current conversation. Pass the skill's `name` "
        "(as listed in the system-reminder block of available skills) and optional `args` "
        "string. The skill's instructions are returned as text — read them and continue "
        "the conversation following those instructions."
    )
    inputSchema = {
        'type': 'object',
        'properties': {
            'skill': {'type': 'string', 'description': 'Name of the skill to execute.'},
            'args':  {'type': 'string', 'description': 'Optional argument string for $ARGUMENTS.'},
        },
        'required': ['skill'],
        'additionalProperties': False,
    }

    def __init__(self, registry: SkillRegistry) -> None:
        self.registry = registry

    async def call(self, input: dict[str, Any], context: ToolContext) -> ToolResult:
        name, args = _read_input(input)

        # 1. Validate name
        if not name or not SKILL_NAME_RE.match(name):
            return ToolResult(
                content=f'Error: invalid skill name. Must match /^[a-zA-Z0-9_-]+$/. Got: {json.dumps(name)}',
                isError=True,
            )

        # 2. Look up
        skill = self.registry.find_skill(name)
        if not skill:
            return ToolResult(content=f'Error: skill "{name}" not found.', isError=True)

        # 3. Guards
        if skill.frontmatter.disableModelInvocation:
            return ToolResult(
                content=f'Error: skill "{name}" has disable-model-invocation: true. '
                         f'Only invocable by user via /{name}.',
                isError=True,
            )
        if skill.frontmatter.hasForkContext:
            return ToolResult(
                content=f'Error: skill "{name}" declares context: fork (needs sub-agent — not implemented).',
                isError=True,
            )

        # 4. Inject allowed-tools into session allow rules
        if skill.frontmatter.allowedTools and context.addSessionAllowRules:
            context.addSessionAllowRules(skill.frontmatter.allowedTools)

        # 5. Substitute + 6. Return
        session_id = context.sessionId or 'unknown-session'
        prompt_text = build_prompt_text(skill, args, session_id)
        return ToolResult(content=(
            f'Loaded skill "{skill.name}" ({skill.source}). '
            f'Follow the instructions below — they ARE your next steps for this turn.\n\n'
            f'{prompt_text}'
        ))

    def is_read_only(self) -> bool: return False  # Plan Mode rejects
    def is_enabled(self) -> bool: return True


### 10.1 Demo — model invokes the `code-reviewer` skill

We register a `Skill` tool against our registry, then simulate the agentic
loop dispatching a `tool_use` with `{ skill: 'code-reviewer', args: 'src/auth.py' }`.
The tool result is the skill's instructions, with `$ARGUMENTS`,
`${CLAUDE_SKILL_DIR}`, and `${CLAUDE_SESSION_ID}` substituted.


In [ ]:
# First add the code-reviewer skill to our demo workspace.
write_skill(DEMO_USER, 'code-reviewer', SAMPLE_SKILL)
REGISTRY = SkillRegistry()
await bootstrap_skills(DEMO_CWD, REGISTRY, user_dir=DEMO_USER)

# Build a ToolContext with an addSessionAllowRules sink so we can see the injection.
session_allow_rules: list[str] = []
def _allow(rules: list[str]) -> None:
    session_allow_rules.extend(rules)

context = ToolContext(
    cwd=str(DEMO_CWD), sessionId='sess_demo_42',
    addSessionAllowRules=_allow,
)

tool = SkillTool(REGISTRY)
result = await tool.call({'skill': 'code-reviewer', 'args': 'src/auth.py'}, context)
print('isError              :', result.isError)
print('session_allow_rules  :', session_allow_rules)
print()
print('───── tool result content ─────')
print(result.content)


### 10.2 Demo — error paths

Invalid name → 400-ish error. Unknown skill → not-found. These error
results carry `isError=True` so the agentic loop relays them back to the
model with the `is_error` flag set on the `tool_result` content block.


In [ ]:
for case in [
    {'skill': 'with spaces!', 'args': ''},
    {'skill': 'never-loaded', 'args': ''},
]:
    r = await tool.call(case, context)
    print(f'input={case}  → isError={r.isError}')
    print(f'  {r.content}')
    print()


## 11. User Slash-Command Expansion — `/skill-name args`

Source: `src/core/queryEngine.ts:tryExpandSkillCommand`.

When the **user** (not the model) types `/code-reviewer src/auth.py`, the
QueryEngine bypasses the model entirely: it does the same lookup, allow-
rule injection, and template substitution that the `Skill` tool does, then
injects the skill's body as a **user message** so the model sees direct
instructions instead of a tool result.

The function also emits `<command-message> / <command-name> /
<command-args>` marker tags so the terminal UI can render the user's input
as a command bubble rather than raw text.


In [ ]:
SKILL_COMMAND_RE = re.compile(r'^/([a-zA-Z0-9_-]+)(?:\s+(.*))?$')

@dataclass
class SkillExpansion:
    skill: Skill
    markerContent: str  # rendered as a command bubble in the UI
    bodyText: str       # injected as a user message into the conversation

def try_expand_skill_command(
    input_str: str, registry: SkillRegistry, context: ToolContext,
) -> Optional[SkillExpansion]:
    match = SKILL_COMMAND_RE.match(input_str)
    if not match:
        return None
    name, raw_args = match.group(1), match.group(2)
    skill = registry.find_skill(name)
    if not skill:
        return None  # fall through to the generic /command dispatcher

    args = (raw_args or '').strip()
    dir_posix = skill.baseDir.replace(os.sep, '/')
    session_id = context.sessionId or 'unknown-session'

    if skill.frontmatter.allowedTools and context.addSessionAllowRules:
        context.addSessionAllowRules(skill.frontmatter.allowedTools)

    body = (skill.body
            .replace('${CLAUDE_SKILL_DIR}', dir_posix)
            .replace('${CLAUDE_SESSION_ID}', session_id)
            .replace('$ARGUMENTS', args))

    marker_lines = [
        f'<command-message>{skill.name}</command-message>',
        f'<command-name>/{skill.name}</command-name>',
    ]
    if args:
        marker_lines.append(f'<command-args>{args}</command-args>')
    marker = '\n'.join(marker_lines)

    header = (
        f'[skill_invocation:{skill.name}]\n'
        f'Run skill "{skill.name}" with the following instructions. '
        f'Base directory for this skill: {dir_posix}.\n\n'
    )
    return SkillExpansion(skill=skill, markerContent=marker, bodyText=header + body)


In [ ]:
# Simulate user typing `/code-reviewer src/auth.py` in the REPL.
expansion = try_expand_skill_command(
    '/code-reviewer src/auth.py', REGISTRY, context,
)
assert expansion is not None
print('───── command marker (UI bubble) ─────')
print(expansion.markerContent)
print('\n───── body injected as user message ─────')
print(expansion.bodyText)

# Non-skill slash → returns None, the engine falls back to /command dispatcher.
print('\n/not-a-skill → ', try_expand_skill_command('/not-a-skill', REGISTRY, context))


## 12. End-to-End — From Boot to Invocation in One Flow

Stitches the pieces together as a single agentic-loop tick would:

1. **Boot** — load disk skills into the registry.
2. **Build the system prompt** — get the `<system-reminder>` listing.
3. **Model picks a skill** — emits a `tool_use` for `Skill`.
4. **Tool result returned** — the substituted skill body.
5. **Model reads a matching file** — the post-tool-call hook activates a
   conditional skill, so the *next* system prompt will list it too.


In [ ]:
# Reset the registry for a clean run.
REGISTRY = SkillRegistry()
await bootstrap_skills(DEMO_CWD, REGISTRY, user_dir=DEMO_USER)

print('─── 1. boot complete ───')
print('dynamic     :', list(REGISTRY.dynamic.keys()))
print('conditional :', list(REGISTRY.conditional.keys()))

print('\n─── 2. system-prompt skill listing ───')
print(format_skills_system_reminder(REGISTRY.get_model_visible_skills()))

print('\n─── 3. model invokes Skill(skill="code-reviewer", args="src/auth.py") ───')
tool = SkillTool(REGISTRY)
session_allow_rules.clear()
result = await tool.call({'skill': 'code-reviewer', 'args': 'src/auth.py'}, context)
print('result.isError    :', result.isError)
print('allow rules added :', session_allow_rules)
print('tool result body (first 400 chars):')
print(result.content[:400], '...')

print('\n─── 4. while following the skill, the model reads a test file ───')
tool_input = {'file_path': 'tests/auth.spec.py'}
activated = activate_conditional_skills_for_paths(
    extract_tool_file_paths('Read', tool_input), DEMO_CWD, REGISTRY,
)
print('newly activated   :', activated)

print('\n─── 5. next system-prompt listing now includes test-reviewer ───')
print(format_skills_system_reminder(REGISTRY.get_model_visible_skills()))


## 13. Cleanup & Summary

Removes the temp workspace, then maps each cell back to its source file.

| Section | Concept | Source file |
|---|---|---|
| 2 | `Skill`, `SkillFrontmatter`, `SkillSource` | `src/types/types.ts` |
| 3 | Minimal `Tool` / `ToolContext` / `ToolResult` | `src/tools/Tool.ts` |
| 4 | `split_frontmatter`, `normalize_frontmatter`, `extract_fallback_description` | `src/services/skills/parseFrontmatter.ts` |
| 5 | `load_from_one_dir`, `load_all_skills`, symlink + name dedup | `src/services/skills/loadSkillsDir.ts` |
| 6 | Two-map registry, `find_skill`, `activate_conditional`, `get_model_visible_skills` | `src/services/skills/registry.ts` |
| 7 | Three-tier budget formatter + `<system-reminder>` block | `src/services/skills/budget.ts` |
| 8 | `bootstrap_skills` startup orchestration | `src/services/skills/bootstrap.ts` |
| 9 | `extract_tool_file_paths`, `activate_conditional_skills_for_paths` | `src/services/skills/conditional.ts` |
| 10 | `SkillTool` — name validation, lookup, guards, substitution | `src/tools/skillTool.ts` |
| 11 | `try_expand_skill_command` user slash-command flow | `src/core/queryEngine.ts` |
| 12 | End-to-end agentic-loop tick | (assembled) |

**What was simplified vs. source:**

- No MCP / sub-agent / sandbox / persistence wiring.
- `pathspec` substitutes for the npm `ignore` package.
- The TS source uses module-level state in `registry.ts`; we use a class
  so the notebook can have isolated registries.
- The `addSessionAllowRules` callback is a list-appender stub; the real
  implementation routes into the permissions module.
- Permission settings / Plan-Mode rejection / observability hooks dropped.


In [ ]:
shutil.rmtree(DEMO_WORKSPACE, ignore_errors=True)
print(f'Removed {DEMO_WORKSPACE}')
